# Precipitation Perturbation Analysis using Aurora-Lite with Decoders

This notebook demonstrates how to apply temperature perturbations and analyze their effects on precipitation predictions using the Aurora-Lite model with hydrological decoders.

In [ ]:
import sys
import os
import numpy as np
import h5py
import xarray as xr
import torch
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from pathlib import Path

# Add the project directory to Python path
project_dir = os.path.abspath('.')
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

from aurora.batch import Batch, Metadata
from aurora.model.aurora_lite import AuroraLite
from aurora.model.decoder_lite import MLPDecoderLite
from transform_data import transform_data

print(f"Working directory: {os.getcwd()}")
print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Setup Perturbation Parameters

Based on the methodology from PerturbationTries.ipynb:
- Temperature perturbation derived from latent heat and specific heat
- Applied to atmospheric pressure levels 700-925 hPa
- Gaussian spatial distribution with configurable center and spread

In [ ]:
# Perturbation parameters (from PerturbationTries.ipynb)
Lv = 2.5e6       # Latent heat of vaporization [J/kg]
cp = 1005        # Specific heat at constant pressure [J/kg·K]
delta_q = 0.0010 # Specific humidity perturbation
delta_T = (Lv * delta_q) / cp  # Temperature perturbation
sigma = 8.0      # Gaussian spread parameter

# Perturbation location (can be modified)
center_lat = 60.0
center_lon = -147.0

# Pressure levels for perturbation
perturb_levels = [700, 850, 925]

print(f"Temperature perturbation amplitude: {delta_T:.3f} K")
print(f"Perturbation center: ({center_lat}, {center_lon})")
print(f"Pressure levels: {perturb_levels} hPa")

## Load Models

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Aurora model
modelAurora = AuroraLite(
    use_lora=False, 
    autocast=True,
    surf_vars=("2t", "10u", "10v", "msl"),
    static_vars=("lsm", "z", "slt"),
    atmos_vars=("z", "u", "v", "t", "q")
)

modelAurora.load_checkpoint("microsoft/aurora", "aurora-0.25-pretrained.ckpt")
modelAurora = modelAurora.to(device)
modelAurora.eval()

print("Aurora model loaded successfully!")

In [ ]:
# Load decoder model
surf_vars_new = ["tp_mswep", "pe", "r", "swc"]

modelDecoder = MLPDecoderLite(
    surf_vars_new=surf_vars_new, 
    patch_size=modelAurora.decoder.patch_size, 
    embed_dim=2*modelAurora.encoder.embed_dim,
    hidden_dims=[512, 512, 256],
)

checkpoint = torch.load("./lite-decoder.ckpt")
modelDecoder.load_state_dict(checkpoint)
modelDecoder.to(device)
modelDecoder.eval()

print("Decoder model loaded successfully!")

## Load Data (2012-10-24)

In [ ]:
download_path = Path("./data/downloads")

# Load datasets
static_vars_ds = xr.open_dataset(download_path / "static.nc", engine="netcdf4")
static_vars_ds = static_vars_ds.sel(latitude=static_vars_ds.latitude[:720])

surf_vars_ds = xr.open_dataset(download_path / "2012-10-24-surface-level.nc", engine="netcdf4")
surf_vars_ds = surf_vars_ds.sel(latitude=surf_vars_ds.latitude[:720])

atmos_vars_ds = xr.open_dataset(download_path / "2012-10-24-atmospheric.nc", engine="netcdf4")
atmos_vars_ds = atmos_vars_ds.sel(latitude=atmos_vars_ds.latitude[:720])

print("Data loaded successfully!")
print(f"Surface data shape: {surf_vars_ds['t2m'].shape}")
print(f"Atmospheric data shape: {atmos_vars_ds['t'].shape}")
print(f"Available pressure levels: {list(atmos_vars_ds.pressure_level.values)}")

## Create Baseline Batch

In [ ]:
# Create baseline batch (no perturbation)
batch_baseline = Batch(
    surf_vars={
        "2t": torch.from_numpy(surf_vars_ds["t2m"].values[:2][None]),
        "10u": torch.from_numpy(surf_vars_ds["u10"].values[:2][None]),
        "10v": torch.from_numpy(surf_vars_ds["v10"].values[:2][None]),
        "msl": torch.from_numpy(surf_vars_ds["msl"].values[:2][None]),
    },
    static_vars={
        "z": torch.from_numpy(static_vars_ds["z"].values[0]),
        "slt": torch.from_numpy(static_vars_ds["slt"].values[0]),
        "lsm": torch.from_numpy(static_vars_ds["lsm"].values[0]),
    },
    atmos_vars={
        "t": torch.from_numpy(atmos_vars_ds["t"].values[:2][None]),
        "u": torch.from_numpy(atmos_vars_ds["u"].values[:2][None]),
        "v": torch.from_numpy(atmos_vars_ds["v"].values[:2][None]),
        "q": torch.from_numpy(atmos_vars_ds["q"].values[:2][None]),
        "z": torch.from_numpy(atmos_vars_ds["z"].values[:2][None]),
    },
    metadata=Metadata(
        lat=torch.from_numpy(surf_vars_ds.latitude.values),
        lon=torch.from_numpy(surf_vars_ds.longitude.values),
        time=(surf_vars_ds.valid_time.values.astype("datetime64[s]").tolist()[1],),
        atmos_levels=tuple(int(level) for level in atmos_vars_ds.pressure_level.values),
    ),
)

print(f"Baseline batch created for time: {batch_baseline.metadata.time[0]}")
print(f"Batch shape - surf_vars: {batch_baseline.surf_vars['2t'].shape}")
print(f"Batch shape - atmos_vars: {batch_baseline.atmos_vars['t'].shape}")

## Create Gaussian Perturbation Mask

In [ ]:
# Get coordinate arrays
lat = batch_baseline.metadata.lat.numpy()
lon = batch_baseline.metadata.lon.numpy()

# Handle longitude wrapping
center_lon_adj = center_lon
if batch_baseline.metadata.lon.max() > 180:
    center_lon_adj = center_lon % 360

print(f"Original center_lon: {center_lon}, adjusted: {center_lon_adj}")
print(f"Longitude range: {lon.min():.1f} to {lon.max():.1f}")
print(f"Latitude range: {lat.min():.1f} to {lat.max():.1f}")

# Create meshgrid
LON, LAT = np.meshgrid(lon, lat)

# Calculate distances
dlon = LON - center_lon_adj
dlat = LAT - center_lat

# Handle longitude wrapping for distance calculation
dlon = np.where(dlon > 180, dlon - 360, dlon)
dlon = np.where(dlon < -180, dlon + 360, dlon)

# Create Gaussian mask
gaussian_mask = delta_T * np.exp(-(dlon**2 + dlat**2) / (2 * sigma**2))
gaussian_mask_torch = torch.from_numpy(gaussian_mask).float()

print(f"Gaussian mask shape: {gaussian_mask.shape}")
print(f"Max perturbation amplitude: {gaussian_mask.max():.3f} K")

## Visualize Perturbation

In [ ]:
# Plot the Gaussian perturbation mask
fig, ax = plt.subplots(figsize=(10, 6))

im = ax.imshow(gaussian_mask, extent=(-180, 180, -90, 90), cmap='Reds', origin='lower')
ax.plot(center_lon, center_lat, 'ko', markersize=10, markerfacecolor='blue')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Temperature Perturbation Mask\nCenter: ({center_lat}, {center_lon}), σ={sigma}')
plt.colorbar(im, ax=ax, label='ΔT (K)')
plt.tight_layout()
plt.show()

print(f"Perturbation center marked in blue at ({center_lat}, {center_lon})")

## Create Perturbed Batch

In [ ]:
# Clone temperature tensor
t_pert = batch_baseline.atmos_vars["t"].clone()  # shape: [B, T, L, H, W]

# Get pressure levels and find indices for perturbation levels
p_levels = torch.tensor(batch_baseline.metadata.atmos_levels)
plev_idx = []

for level in perturb_levels:
    try:
        idx = torch.where(p_levels == level)[0][0]
        plev_idx.append(idx)
        print(f"Pressure level {level} hPa found at index {idx}")
    except IndexError:
        print(f"Warning: Pressure level {level} hPa not found")

# Apply perturbation to selected pressure levels
for i in plev_idx:
    t_pert[0, :, i] += gaussian_mask_torch  # [B=0, T=all, L=i, H, W]
    print(f"Applied perturbation to level index {i}")

# Create perturbed batch
batch_perturbed = Batch(
    surf_vars=batch_baseline.surf_vars,
    static_vars=batch_baseline.static_vars,
    atmos_vars={
        "t": t_pert,  # Use perturbed temperature
        "u": batch_baseline.atmos_vars["u"],
        "v": batch_baseline.atmos_vars["v"],
        "q": batch_baseline.atmos_vars["q"],
        "z": batch_baseline.atmos_vars["z"],
    },
    metadata=batch_baseline.metadata,
)

print("Perturbed batch created successfully!")
print(f"Temperature difference max: {torch.max(t_pert - batch_baseline.atmos_vars['t']):.3f} K")
print(f"Temperature difference min: {torch.min(t_pert - batch_baseline.atmos_vars['t']):.3f} K")

## Run Baseline Prediction

In [ ]:
print("Running baseline prediction...")

with torch.inference_mode():
    # Run Aurora prediction
    preds_baseline_org, lat_dec_baseline = modelAurora.forward(batch_baseline)
    
    # Run decoder prediction
    latent_decoder_baseline = lat_dec_baseline.detach().clone()
    preds_baseline_new = modelDecoder.forward(
        latent_decoder_baseline, 
        batch_baseline.metadata.lat, 
        batch_baseline.metadata.lon
    )

# Transform decoder predictions back to original scale
preds_baseline_transformed = {
    k: transform_data(v.cpu().numpy().squeeze(), k, direct=False) 
    for k, v in preds_baseline_new.items()
}

print("Baseline prediction completed!")
print(f"Available predictions: {list(preds_baseline_transformed.keys())}")
print(f"Precipitation shape: {preds_baseline_transformed['tp_mswep'].shape}")
print(f"Precipitation stats: min={preds_baseline_transformed['tp_mswep'].min():.4f}, max={preds_baseline_transformed['tp_mswep'].max():.4f}")

## Run Perturbed Prediction

In [ ]:
print("Running perturbed prediction...")

with torch.inference_mode():
    # Run Aurora prediction with perturbation
    preds_perturbed_org, lat_dec_perturbed = modelAurora.forward(batch_perturbed)
    
    # Run decoder prediction
    latent_decoder_perturbed = lat_dec_perturbed.detach().clone()
    preds_perturbed_new = modelDecoder.forward(
        latent_decoder_perturbed, 
        batch_perturbed.metadata.lat, 
        batch_perturbed.metadata.lon
    )

# Transform decoder predictions back to original scale
preds_perturbed_transformed = {
    k: transform_data(v.cpu().numpy().squeeze(), k, direct=False) 
    for k, v in preds_perturbed_new.items()
}

print("Perturbed prediction completed!")
print(f"Precipitation shape: {preds_perturbed_transformed['tp_mswep'].shape}")
print(f"Precipitation stats: min={preds_perturbed_transformed['tp_mswep'].min():.4f}, max={preds_perturbed_transformed['tp_mswep'].max():.4f}")

## Analyze Precipitation Changes

In [ ]:
# Extract precipitation data
baseline_precip = preds_baseline_transformed['tp_mswep']
perturbed_precip = preds_perturbed_transformed['tp_mswep']

# Calculate differences
precip_diff = perturbed_precip - baseline_precip

# Calculate statistics
stats = {
    'max_increase': np.max(precip_diff),
    'max_decrease': np.min(precip_diff),
    'mean_change': np.mean(precip_diff),
    'std_change': np.std(precip_diff),
    'total_baseline': np.sum(baseline_precip),
    'total_perturbed': np.sum(perturbed_precip),
    'relative_change': (np.sum(perturbed_precip) - np.sum(baseline_precip)) / np.sum(baseline_precip) * 100
}

print("Precipitation Change Analysis:")
print("=" * 40)
print(f"Max increase: {stats['max_increase']:.6f} m/6h ({stats['max_increase']*1000:.3f} mm/6h)")
print(f"Max decrease: {stats['max_decrease']:.6f} m/6h ({stats['max_decrease']*1000:.3f} mm/6h)")
print(f"Mean change: {stats['mean_change']:.6f} m/6h ({stats['mean_change']*1000:.6f} mm/6h)")
print(f"Std change: {stats['std_change']:.6f} m/6h ({stats['std_change']*1000:.6f} mm/6h)")
print(f"Total baseline: {stats['total_baseline']:.3f} m ({stats['total_baseline']*1000:.3f} mm)")
print(f"Total perturbed: {stats['total_perturbed']:.3f} m ({stats['total_perturbed']*1000:.3f} mm)")
print(f"Relative change: {stats['relative_change']:.4f}%")

## Visualize Results

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Common parameters
extent = (-180, 180, -90, 90)
precip_vmin, precip_vmax = 1e-6, 0.1  # in meters
diff_vmax = np.max(np.abs(precip_diff))

# 1. Baseline precipitation
im1 = axes[0, 0].imshow(
    baseline_precip * 1000,  # convert to mm
    extent=extent, cmap='Blues', 
    norm=plt.LogNorm(vmin=precip_vmin*1000, vmax=precip_vmax*1000),
    origin='lower'
)
axes[0, 0].set_title('Baseline Precipitation (mm/6h)')
axes[0, 0].set_xlabel('Longitude')
axes[0, 0].set_ylabel('Latitude')
plt.colorbar(im1, ax=axes[0, 0], shrink=0.8)

# 2. Perturbed precipitation
im2 = axes[0, 1].imshow(
    perturbed_precip * 1000,  # convert to mm
    extent=extent, cmap='Blues', 
    norm=plt.LogNorm(vmin=precip_vmin*1000, vmax=precip_vmax*1000),
    origin='lower'
)
axes[0, 1].set_title('Perturbed Precipitation (mm/6h)')
axes[0, 1].set_xlabel('Longitude')
axes[0, 1].set_ylabel('Latitude')
plt.colorbar(im2, ax=axes[0, 1], shrink=0.8)

# 3. Precipitation difference
im3 = axes[1, 0].imshow(
    precip_diff * 1000,  # convert to mm
    extent=extent, cmap='RdBu_r',
    vmin=-diff_vmax*1000, vmax=diff_vmax*1000,
    origin='lower'
)
axes[1, 0].set_title('Precipitation Difference (mm/6h)')
axes[1, 0].set_xlabel('Longitude')
axes[1, 0].set_ylabel('Latitude')
plt.colorbar(im3, ax=axes[1, 0], shrink=0.8)

# 4. Temperature perturbation
im4 = axes[1, 1].imshow(
    gaussian_mask, extent=extent, cmap='Reds',
    origin='lower'
)
axes[1, 1].plot(center_lon, center_lat, 'ko', markersize=10, markerfacecolor='blue')
axes[1, 1].set_title(f'Temperature Perturbation\nCenter: ({center_lat}, {center_lon})')
axes[1, 1].set_xlabel('Longitude')
axes[1, 1].set_ylabel('Latitude')
plt.colorbar(im4, ax=axes[1, 1], shrink=0.8, label='ΔT (K)')

plt.tight_layout()
plt.show()

print(f"Perturbation center marked in blue at ({center_lat}, {center_lon})")

## Additional Analysis: Regional Focus

In [ ]:
# Focus on the perturbation region
lat_range = slice(int(center_lat - 20), int(center_lat + 20))
lon_range = slice(int((center_lon + 180) - 30), int((center_lon + 180) + 30))  # Handle longitude indexing

print(f"Focusing on region around perturbation center...")
print(f"Latitude range: {center_lat - 20} to {center_lat + 20}")
print(f"Longitude range: {center_lon - 30} to {center_lon + 30}")

# Extract regional data
baseline_precip_region = baseline_precip[lat_range, lon_range]
perturbed_precip_region = perturbed_precip[lat_range, lon_range]
precip_diff_region = precip_diff[lat_range, lon_range]

# Regional statistics
regional_stats = {
    'max_increase': np.max(precip_diff_region),
    'max_decrease': np.min(precip_diff_region),
    'mean_change': np.mean(precip_diff_region),
    'std_change': np.std(precip_diff_region),
}

print("\nRegional Precipitation Change Statistics:")
print("=" * 40)
print(f"Max increase: {regional_stats['max_increase']*1000:.3f} mm/6h")
print(f"Max decrease: {regional_stats['max_decrease']*1000:.3f} mm/6h")
print(f"Mean change: {regional_stats['mean_change']*1000:.6f} mm/6h")
print(f"Std change: {regional_stats['std_change']*1000:.6f} mm/6h")

## Summary and Conclusions

In [ ]:
print("Precipitation Perturbation Analysis Summary")
print("=" * 50)
print(f"Date analyzed: 2012-10-24")
print(f"Perturbation location: ({center_lat}, {center_lon})")
print(f"Perturbation amplitude: {delta_T:.3f} K")
print(f"Pressure levels affected: {perturb_levels} hPa")
print(f"Gaussian spread (σ): {sigma}")
print()
print("Key Findings:")
print(f"• Maximum precipitation increase: {stats['max_increase']*1000:.3f} mm/6h")
print(f"• Maximum precipitation decrease: {abs(stats['max_decrease'])*1000:.3f} mm/6h")
print(f"• Overall precipitation change: {stats['relative_change']:.4f}%")
print(f"• Standard deviation of changes: {stats['std_change']*1000:.6f} mm/6h")
print()
print("This analysis demonstrates how temperature perturbations at mid-to-lower")
print("atmospheric levels can influence precipitation patterns through the")
print("Aurora-Lite model with hydrological decoders.")